# Chapter 9 &mdash; Implementing State Elimination: Corner Cases

**Concept 7 of the Chapter 9 decomposition:** *Implementing State Elimination: `del_gnfa_states` and its Corner Cases*

A doubly-nested loop over all $(p,q)$ pairs for each deleted state &mdash; with plenty of corner cases.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter9/Concept-Implementing-State-Elimination/Concept-Implementing-State-Elimination.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.Def_NFA2RE     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


`del_gnfa_states` is a loop over states to delete; inside it, `del_one_gnfa_state`
loops over all $(p,q)$ pairs. The corner cases are what make it fiddly:

* **no self-loop** &mdash; the starred middle is $\varepsilon$, not $\emptyset$;
* **no edge $p\to s$ or $s\to q$** &mdash; skip the pair entirely;
* **an existing $p\to q$ edge** &mdash; union, do not overwrite;
* **$p = q$** &mdash; the bypass becomes a *new self-loop* on $p$;
* **the last two states** &mdash; stop; `Real_I` and `Real_F` are never deleted.

`Edges_Exist_Via(G,p,q)` is the predicate that guards most of these.

## 2. Definitions

### The helpers Jove exposes

In [ ]:
import inspect
for f in [del_gnfa_states, del_one_gnfa_state, choose_state_to_del, Edges_Exist_Via]:
    print("%-22s %s" % (f.__name__, inspect.signature(f)))

### Machines that exercise each corner case

In [ ]:
def widen(D, sig):
    # give D the alphabet sig, then totalize, so two machines whose REs
    # mention different symbols can still be compared
    extra = sig - D["Sigma"]
    return totalize_dfa(addtosigma_dfa(D, extra) if extra else D)

def same_language(D1, D2):
    sig = D1["Sigma"] | D2["Sigma"]
    A, B = min_dfa(widen(D1, sig)), min_dfa(widen(D2, sig))
    return langeq_dfa(A, B) and iso_dfa(A, B)

CASES = {
 'no self-loop'   : '''NFA
I : 0 -> S
S : 1 -> F
''',
 'with self-loop' : '''NFA
I : 0 -> S
S : 1 -> S
S : 0 -> F
''',
 'existing edge'  : '''NFA
I : 0 -> S
I : 1 -> F
S : 0 -> F
''',
 'p equals q'     : '''NFA
I : 0 -> S
S : 1 -> I
I : 1 -> F
''',
 'dead end'       : '''NFA
I : 0 -> S
I : 1 -> F
S : 0 -> S
''',
}

## 3. Tests

`Edges_Exist_Via` is the guard: it returns the label list, or `False`.

In [ ]:
N = md2mc(CASES['no self-loop'])
g = mk_gnfa(N)
for p in sorted(g["Q"]):
    for q in sorted(g["Q"]):
        e = Edges_Exist_Via(g, p, q)
        if e: print("   %-8s -> %-8s : %s" % (p, q, e))

Every corner case converts, and every one round-trips.

In [ ]:
for name, src in CASES.items():
    N = md2mc(src)
    _, _, r = del_gnfa_states(mk_gnfa(N))
    D0, D1 = min_dfa(nfa2dfa(N)), min_dfa(nfa2dfa(re2nfa(r)))
    print("%-16s RE %-40s same language %s" % (name, r[:38], same_language(D0, D1)))
    assert same_language(D0, D1)

**No self-loop** must give $\varepsilon$ in the middle, not $\emptyset$.

In [ ]:
N = md2mc(CASES['no self-loop'])
_, _, r = del_gnfa_states(mk_gnfa(N))
D = min_dfa(nfa2dfa(re2nfa(r)))
print("RE :", r, "  accepts '01'?", accepts_dfa(D, '01'))
assert accepts_dfa(D, '01')
print("if the middle had been the empty SET, nothing would be accepted.")

**$p = q$** turns a bypass into a new self-loop, which the next step must star.

In [ ]:
N = md2mc(CASES['p equals q'])
_, _, r = del_gnfa_states(mk_gnfa(N))
D = min_dfa(nfa2dfa(re2nfa(r)))
for s in ['1', '011', '01011', '0101011']:
    print("   %-10r accepted? %s" % (s, accepts_dfa(D, s)))
assert accepts_dfa(D, '01011') and accepts_dfa(D, '1')
print("\nthe (01)* loop survived deletion as a self-loop on I")

**Dead end:** a state with no path to `Real_F` contributes nothing.

In [ ]:
N = md2mc(CASES['dead end'])
_, _, r = del_gnfa_states(mk_gnfa(N))
print("RE :", repr(r), "  -- the 0-edges vanished entirely")
D = min_dfa(nfa2dfa(re2nfa(r)))
print("alphabet of the RE's machine :", sorted(D["Sigma"]),
      " (the original had", sorted(N["Sigma"]), ")")
Dw = min_dfa(widen(D, N["Sigma"]))
from itertools import product
acc = [''.join(p) for k in range(5) for p in product('01', repeat=k)
       if accepts_dfa(Dw, ''.join(p))]
print("accepted :", acc)
assert acc == ['1']
print("S can never reach a final state, so it contributes no alternatives --")
print("and with it goes every mention of 0 in the expression.")

`choose_state_to_del` picks a low-degree state, which keeps labels short.

In [ ]:
N = md2mc(CASES['with self-loop'])
g = mk_gnfa(N)
left = sorted(g["Q"] - {"Real_I", "Real_F"})
print("candidates :", left, " -> picked", choose_state_to_del(g, left))

## 4. Exercises


1. Which corner case would you most likely get wrong writing this yourself?
2. Why is $\varepsilon$ (not $\emptyset$) the right default for a missing self-loop?
3. Add a fifth corner case of your own and check it round-trips.

In [ ]:
# Your work for the exercises above.